# The orbit — how it works

This is the explanation and reference for `quicksat.utils.orbit.Orbit`: the one orbit every budget shares, what it derives, and what it refuses to model.

It is deliberately not a tutorial. In Diátaxis terms, `sample/` holds the tutorials and how-to guides; `docs/` holds the explanation and the reference. It still runs, against the same sample data, because an explanation that cannot be executed drifts from the code it describes.

The budget references — `data_budget_ref.ipynb`, and the delta-V and agility ones when they exist — each assume the orbit and point here rather than restating it.

In [1]:
import os
from pathlib import Path

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import MU_EARTH, Q_, R_EARTH
from quicksat.utils.orbit import Orbit

## One orbit, stated once

Three budgets need the orbit, and each needs something different from it:

| budget | needs |
|---|---|
| data | the period, and how many orbits fit in a day |
| delta-V | the circular velocity, for Hohmann transfers and plane changes |
| agility | the ground track speed, to turn a slew time into a distance on the ground |

All three derive from one number — the altitude. Writing that number into three config files would work exactly until someone retuned one of them, at which point the budgets would quietly disagree about what satellite they were describing. So the altitude lives in `sample/data/orbit.yaml`, outside any one budget's config, and the derivations live in one class rather than three copies of Kepler's third law.

In [2]:
orbit = Orbit.from_yaml_file("sample/data/orbit.yaml")
print(Path("sample/data/orbit.yaml").read_text())
orbit

# Shared orbit. Read by every module that needs it: the data
# budget for period and orbits per day, delta-V for its Hohmann transfers,
# agility for ground track speed.
#
# Circular throughout. quicksat models no eccentricity.

altitude: 500 km
inclination: 97.4 deg      # sun-synchronous at this altitude



Orbit(altitude=<Quantity(500, 'kilometer')>, inclination=<Quantity(97.4, 'degree')>)

## The file

Two settings, both unit-bearing text parsed with pint on load.

| setting | meaning | checked |
|---|---|---|
| `altitude` | height above the WGS-84 equatorial radius | must be a non-negative length |
| `inclination` | orbit plane inclination | must carry no dimension other than angle |

Nothing derives from `inclination` yet. It is read by the plane-change calculations in the delta-V budget, which do not exist, so for now it is carried through loading and validation and no further. It is in the file because the file describes the orbit, not because anything currently consumes it.

The altitude check is a real one: `500 kg` is rejected at load rather than becoming a number that happens to be wrong. The inclination check is weaker, and the reason is in the units section below.

In [3]:
for text in (
    "altitude: 500 kg\ninclination: 97.4 deg",
    "altitude: -500 km\ninclination: 97.4 deg",
    "altitude: 500 km\ninclination: 97.4 km",
    "inclination: 97.4 deg",
):
    label = text.replace("\n", ", ")
    try:
        Orbit.from_yaml_text(text)
        print(f"{label:46s} accepted")
    except ValueError as exc:
        reason = str(exc).splitlines()[2].strip().split(" [type=")[0]
        print(f"{label:46s} rejected - {reason}")

altitude: 500 kg, inclination: 97.4 deg        rejected - Value error, Value must have length dimensions, got '500 kilogram'
altitude: -500 km, inclination: 97.4 deg       rejected - Value error, Length must not be negative, got '-500 kilometer'
altitude: 500 km, inclination: 97.4 km         rejected - Value error, Value must have angle dimensions, got '97.4 kilometer'
inclination: 97.4 deg                          rejected - Field required


## Earth constants

pint ships `standard_gravity`, which is what the rocket equation needs, but it has no Earth radius and no gravitational parameter. `quicksat` adds both to the shared registry, so they are quantities like any other rather than floats scattered through the code:

```python
u.define("earth_radius = 6378.137 km = R_earth")            # WGS-84 equatorial
u.define("earth_mu = 398600.4418 km**3 / s**2 = GM_earth")  # EGM96
```

**`earth_mu` is the measured GM, not `gravitational_constant × earth_mass`.** That is not pedantry. $G$ is among the worst-measured constants in physics — known to roughly five significant figures — while the *product* $GM$ is measured directly from spacecraft tracking to about nine. Deriving μ from $G$ and a mass would throw four significant figures away for no reason, and the error would land in every period and every velocity the tool computes.

In [4]:
print(f"R_EARTH   {R_EARTH.to('km'):~.3f}")
print(f"MU_EARTH  {MU_EARTH.to('km**3/s**2'):~.4f}")
print(f"\npint has standard_gravity: {Q_(1, 'standard_gravity'):~}")
print(f"pint has gravitational_constant: {Q_(1, 'gravitational_constant').to('m**3/(kg*s**2)'):~.5g}")

# what deriving mu from G and a mass would cost. G is known to about 2.2e-5
# relative; the IAU Earth mass is itself derived from GM, so this is illustrative
G = Q_(1, "gravitational_constant")
earth_mass = Q_(5.9722e24, "kg")
derived = (G * earth_mass).to("km**3/s**2")
print(f"\nG x M      {derived:~.4f}")
print(f"measured   {MU_EARTH.to('km**3/s**2'):~.4f}")
print(f"difference {abs(derived - MU_EARTH).to('km**3/s**2'):~.4f}"
      f"  ({abs(derived / MU_EARTH - 1).to('dimensionless').magnitude * 1e6:.1f} ppm)")

R_EARTH   6378.137 km
MU_EARTH  398600.4418 km ** 3 / s ** 2

pint has standard_gravity: 1 g_0
pint has gravitational_constant: 6.6743e-11 m ** 3 / kg / s ** 2

G x M      398602.5446 km ** 3 / s ** 2
measured   398600.4418 km ** 3 / s ** 2
difference 2.1028 km ** 3 / s ** 2  (5.3 ppm)


## What it derives

Circular orbits only. Given $r = R_\\oplus + h$:

$$T = 2\\pi\\sqrt{\\frac{r^3}{\\mu}} \\qquad v = \\sqrt{\\frac{\\mu}{r}} \\qquad v_{\\text{ground}} = v\\,\\frac{R_\\oplus}{r}$$

`orbits_per_day` is deliberately dimensionless rather than a rate. Its only job is to turn a per-orbit quantity into a per-day one, and a dimensionless multiplier does that without pint objecting that orbits are not seconds.

In [5]:
print(f"altitude            {orbit.altitude:~.0f}")
print(f"radius              {orbit.radius:~.3f}")
print(f"period              {orbit.period:~.1f}   ({orbit.period.to('min'):~.2f})")
print(f"orbits per day      {orbit.orbits_per_day:~.4f}")
print(f"velocity            {orbit.velocity:~.4f}")
print(f"ground track speed  {orbit.ground_track_speed:~.4f}")

# the period and the orbit count are reciprocal by construction
print(f"\nperiod x orbits/day = {(orbit.period * orbit.orbits_per_day).to('day'):~.6f}")

altitude            500 km
radius              6878.137 km
period              5677.0 s   (94.62 min)
orbits per day      15.2194
velocity            7.6126 km / s
ground track speed  7.0592 km / s

period x orbits/day = 1.000000 d


### Ground track speed

The sub-satellite point moves more slowly than the spacecraft, because it traces a smaller circle: the orbit has radius $r$, the ground track has radius $R_\\oplus$. The speed scales by that ratio.

**Earth rotation is not included.** This is the speed along the ground track over a non-rotating Earth, not the speed relative to a fixed point on the surface. For a near-polar orbit the two differ by a few hundred metres per second depending on latitude, which matters for a ground station pass and does not matter for the agility budget's "how far did we travel during that slew" — which is what it is there for.

In [6]:
ratio = (R_EARTH / orbit.radius).to("dimensionless")
print(f"orbit radius {orbit.radius:~.0f} vs Earth radius {R_EARTH.to('km'):~.0f}")
print(f"ratio {ratio:~.4f}, so the ground track runs {(1 - ratio.magnitude) * 100:.1f}% slower")
print(f"  orbital     {orbit.velocity:~.4f}")
print(f"  ground      {orbit.ground_track_speed:~.4f}")

# what that means for an agility case: distance covered during a 60 s slew
slew = Q_(60, "s")
print(f"\nduring a {slew:~.0f} slew the ground track advances "
      f"{(orbit.ground_track_speed * slew).to('km'):~.0f}")

orbit radius 6878 km vs Earth radius 6378 km
ratio 0.9273, so the ground track runs 7.3% slower
  orbital     7.6126 km / s
  ground      7.0592 km / s

during a 60 s slew the ground track advances 424 km


## Units

The altitude may be written in whatever unit suits, and is converted once on load. Everything derived from it is unaffected — which is the point of parsing units rather than documenting them in a comment.

The inclination check is weaker than the altitude check, and the reason is a property of pint rather than a choice: **pint treats radians as dimensionless**, so an angle has no dimension of its own. `AngleQty` can reject `97.4 km`, because that has length dimensions, but it cannot tell `97.4 deg` from a bare `97.4`. A file that omits the unit will be accepted and read as radians.

In [7]:
same = [
    Orbit.from_yaml_text(f"altitude: {text}\ninclination: 97.4 deg")
    for text in ("500 km", "500000 m", "0.5 Mm")
]
for o, text in zip(same, ("500 km", "500000 m", "0.5 Mm"), strict=True):
    print(f"{text:10s} -> altitude {o.altitude.to('km'):~.1f}, period {o.period:~.1f}")

print("\nangles are dimensionless in pint, so this cannot be caught:")
bare = Orbit.from_yaml_text("altitude: 500 km\ninclination: 97.4")
print(f"  'inclination: 97.4' accepted, read as {bare.inclination:~} "
      f"= {bare.inclination.to('deg'):~.1f}")

500 km     -> altitude 500.0 km, period 5677.0 s
500000 m   -> altitude 500.0 km, period 5677.0 s
0.5 Mm     -> altitude 500.0 km, period 5677.0 s

angles are dimensionless in pint, so this cannot be caught:
  'inclination: 97.4' accepted, read as 97.4 = 5580.6 deg


## Limitations

What the orbit model deliberately does not do:

- **Circular only.** No eccentricity, so no apogee, perigee or true anomaly. A Hohmann transfer in the delta-V budget is computed between two circular orbits rather than along an ellipse.
- **No perturbations.** No J2, so no nodal regression and no argument-of-perigee drift. Sun-synchronicity is an assertion in the config file, not something the tool checks or maintains.
- **No drag, and no epoch.** The altitude does not decay, so a mission-end orbit is the same as a mission-start one. There is no date anywhere in the model.
- **Earth only.** `earth_radius` and `earth_mu` are baked into the derivations rather than passed in as a central body.
- **A sphere, for the radius.** `earth_radius` is the WGS-84 *equatorial* radius, so a polar orbit's true altitude above the surface varies by about 21 km over a revolution. Immaterial for sizing; wrong for anything pointing-related.
- **Inclination is carried, not used.** Nothing derives from it yet, and it cannot be fully validated.